In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np
import os
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
from torch import Tensor
import random


In [39]:
image_name_to_person_id: dict[str, int] = dict()
person_id_to_image_names: dict[int, list[str]] = dict()

with open("identity_CelebA.txt", "r") as lines:
    for line in lines:
        image_name, person_id_str = line.split(" ")
        image_name: str = image_name.strip()
        person_id: int = int(person_id_str.strip())
        image_name_to_person_id[image_name] = person_id

        if person_id not in person_id_to_image_names:
            person_id_to_image_names[person_id] = []
        person_id_to_image_names[person_id].append(image_name)

In [40]:
def get_ids_images(
    image_names: list[str], N: int, exclude_images: set[str] = set()
) -> list[tuple[int, str]]:
    ids_images: list[tuple[int, str]] = []

    taken_indexes = set()
    while len(ids_images) < N:
        while True:
            index = np.random.randint(0, len(image_names))
            if index not in taken_indexes:
                image_name = image_names[index]
                if image_name in exclude_images:
                    continue
                taken_indexes.add(index)
                break
        person_id = image_name_to_person_id[image_name]

        if len(person_id_to_image_names[person_id]) < 2:
            continue

        person_images: list[str] = person_id_to_image_names[person_id]

        needed = N - len(ids_images)
        if needed > 5:
            needed = 5
        ids_images.extend(
            list(map(lambda p_img: (person_id, p_img), person_images))[:needed])
        
    return ids_images

def get_training_data(image_names: list[str], N: int, exclude_images: set[str] = set()) -> tuple[list[tuple[str, str, int]], set[int]]:
    ids_images: list[tuple[int, str]] = get_ids_images(image_names, N * 2, exclude_images)

    train_data: set[tuple[str, str, int]] = set()

    ids_used: set[int] = set()

    while len(train_data) < N:
        person_id, img1_name = ids_images.pop(
            np.random.randint(0, len(ids_images)))
                
        img2_name = None
        img2_index = None
        img2_id = None

        while img2_name is None or img2_name == img1_name:
            img2_index = np.random.randint(0, len(ids_images))
            img2_name = ids_images[img2_index][1]
            img2_id = ids_images[img2_index][0]

        label = 1 if person_id == img2_id else 0

        assert img2_id

        ids_used.add(img2_id)
        ids_used.add(person_id)

        train_data.add((img1_name, img2_name, label))

    return list(train_data), ids_used

In [41]:
from typing import Literal


DATA_DIR = "img_align_celeba/"

result_type = Literal["Same person"] | Literal["Different people"]

class FaceVerificationMLP(nn.Module):
    '''Define the MLP model that takes the difference between two embeddings'''
    def __init__(self, input_dim=512):
        super(FaceVerificationMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            # Output: [same_person_prob, different_person_prob]
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)
   

# Optional Preprocessing: Resize, smooth, convert to tensor, and normalize the image if they require it in one of the tasks. Note: this is a sample transoformation, modify it accoringly to the task
default_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
]) 

# Load MTCNN for face detection, note: you should adjust the size wrt data and model, it can be done directly here or in the transform defined above
mtcnn = MTCNN(image_size=160, margin=20)

# Load FaceNet for embedding extraction (we'll be using a pretrained model)
facenet: InceptionResnetV1 = InceptionResnetV1(pretrained='vggface2').eval()
    

def get_embedding(image_path):
    '''Function to get face embedding from an image path'''
    
    img = Image.open(DATA_DIR + image_path).convert('RGB')
    # When needed add the preprocessing img = transform(img)
    face = mtcnn(img)  # returns a cropped, aligned face
    if face is None:
        raise ValueError(f"No face detected in {image_path}")
    face_embedding = facenet(face.unsqueeze(0))  # Add batch dimension
    return face_embedding.detach()


def get_diff_vector(img1_path, img2_path) -> Tensor:
    '''Function to compare two images and get the absolute difference vector'''
    
    emb1 = get_embedding(img1_path)
    emb2 = get_embedding(img2_path)
    return torch.abs(emb1 - emb2)


def predict_same_person(img1_path, img2_path, model) -> result_type:
    '''Prediction Example, for additional experiments you may want to return the decision in numeric form or add model certenity'''
    model.eval()
    diff = get_diff_vector(img1_path, img2_path)
    output = model(diff)
    _, predicted = torch.max(output, 1)
    return 'Same person' if predicted.item() == 1 else 'Different people'


def augment_image(image, augment_type="gaussian_noise"):
    '''Example augmentation function'''
    if augment_type == "gaussian_noise":
        # Add Gaussian noise
        image_np = np.array(image).astype(np.float32)
        # Modify the parameter value to adjust noise level if needed
        noise = np.random.normal(0, 25, image_np.shape)
        noisy_image = image_np + noise
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
        augmented = Image.fromarray(noisy_image)
    elif augment_type == "blur":
        # Apply Gaussian blur with radius 3
        augmented = image.filter(ImageFilter.GaussianBlur(radius=3))
    elif augment_type == "increased_lighting":
        # Increase brightness by 50%
        enhancer = ImageEnhance.Brightness(image)
        augmented = enhancer.enhance(1.5)
    else:
        augmented = image
    return augmented


image_names: list[str] = os.listdir(DATA_DIR)

In [42]:
def train_model(N):
    train_data, ids_used = get_training_data(image_names, N)

    # Prepare training tensors
    X_train = []
    y_train = []
    for img1, img2, label in train_data:
        diff = get_diff_vector(img1, img2)
        X_train.append(diff.squeeze(0))
        y_train.append(label)

    X_train = torch.stack(X_train)
    y_train = torch.tensor(y_train)

    # Define model, loss, optimizer (note: this is an example, adjust it according to the task)
    model = FaceVerificationMLP()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Train the MLP model (note: this is an example, adjust the number of epochs and additional stop criteria to the task in the Assignement 5 list)
    epochs = 20
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")
        
    return model

In [43]:
model_10: FaceVerificationMLP = train_model(10)


models: list[FaceVerificationMLP] = [model_10]

for model in models:
    for i in range(5):
        img1: str = random.choice(image_names)
        img2: str = random.choice(image_names)
        
        person1_id: int = image_name_to_person_id[img1]
        person2_id: int = image_name_to_person_id[img2]
        
        result: result_type = predict_same_person('000001.jpg', '000002.jpg', model)
        print("Prediction result:", result)
        same_person: bool = person1_id == person2_id
        print("Correct" if same_person == (result == 'Same person') else "Incorrect")
    
    for i in range(5):
        id: int = random.choice(list(person_id_to_image_names.keys()))
        
        img1 = random.choice(person_id_to_image_names[id])        
        img2 = random.choice(person_id_to_image_names[id])
        
        person2_id = image_name_to_person_id[img2]
        
        result: result_type = predict_same_person('000001.jpg', '000002.jpg', model)
        print("Prediction result:", result)
        print("Correct" if result == 'Same person' else "Incorrect")
        
    try:
        result: result_type = predict_same_person('000001.jpg', '000002.jpg', model)
        print("Prediction result:", result)
    except ValueError as e:
        print("Error:", e)

Epoch 1/20, Loss: 0.6702
Epoch 2/20, Loss: 0.6519
Epoch 3/20, Loss: 0.6369
Epoch 4/20, Loss: 0.6211
Epoch 5/20, Loss: 0.6032
Epoch 6/20, Loss: 0.5822
Epoch 7/20, Loss: 0.5586
Epoch 8/20, Loss: 0.5329
Epoch 9/20, Loss: 0.5047
Epoch 10/20, Loss: 0.4746
Epoch 11/20, Loss: 0.4438
Epoch 12/20, Loss: 0.4139
Epoch 13/20, Loss: 0.3864
Epoch 14/20, Loss: 0.3627
Epoch 15/20, Loss: 0.3441
Epoch 16/20, Loss: 0.3300
Epoch 17/20, Loss: 0.3208
Epoch 18/20, Loss: 0.3158
Epoch 19/20, Loss: 0.3139
Epoch 20/20, Loss: 0.3136
Prediction result: Different people
Correct
Prediction result: Different people
Correct
Prediction result: Different people
Correct
Prediction result: Different people
Correct
Prediction result: Different people
Correct
Prediction result: Different people
Incorrect
Prediction result: Different people
Incorrect
Prediction result: Different people
Incorrect
Prediction result: Different people
Incorrect
Prediction result: Different people
Incorrect
Prediction result: Different people
